# Prototype: whodunit

Reproduction check for `detectors/whodunit.py`. Not run by the pipeline: upload to Colab (File > Upload notebook).

**Runtime:** CPU runtime is enough.

**Decides:** Reproduce Whodunit Table 4: F1 0.91 (All, 136 features) and 0.89 (Non-Gameable), within 0.02. Cell 6 checks that feature extraction on this Python matches the shipped CSVs; that is the gate for score().

Run top to bottom, then paste the printed comparison back into the chat.

**Note:** cell 3 `exec`s the authors' own feature-extraction code from their Zenodo notebooks (md5-pinned).

In [ ]:
# %% [1] Install + versions. Only radon is pinned: the package's numpy<1.25 / sklearn==1.2.2 pins do not install on Py3.12.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "radon==6.0.1"], check=True)
import hashlib, json, os, zipfile, urllib.request
from importlib.metadata import version
import numpy as np, pandas as pd, xgboost
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
print(sys.version.split()[0], {p: version(p) for p in ["xgboost", "scikit-learn", "pandas", "numpy", "radon"]})

In [ ]:
# %% [2] Fetch the pinned Zenodo version 18834715 (concept DOI 10.5281/zenodo.10152237 now redirects here)
REC = "https://zenodo.org/api/records/18834715/files/{}/content"
FILES = {"_01_datasets.zip": "9d48858d99d63b4f47e1fec103377ae8",          # 27.2 MB: code files + feature CSVs
         "_02_data_collection.zip": "34d13fe2129ad1c8d5e153b8a3040dca",   # remove_all_comments.ipynb
         "_03_feature_extraction.zip": "ae4cae0460b58863304048432261109e"}  # extract_*_features.ipynb
ROOT = "/content/whodunit"
os.makedirs(ROOT, exist_ok=True)
for name, md5 in FILES.items():
    path = f"{ROOT}/{name}"
    if not os.path.exists(path):
        urllib.request.urlretrieve(REC.format(name), path)
    assert hashlib.md5(open(path, "rb").read()).hexdigest() == md5, name
    zipfile.ZipFile(path).extractall(ROOT)
CC = f"{ROOT}/_01_datasets/codechef"
zipfile.ZipFile(f"{CC}/final_files.zip").extractall(CC)
COMBINED = f"{CC}/final_files_with_comments/unverified/combined"   # 1596 raw files, used for RQ1/RQ2
FEAT = f"{CC}/features"

In [ ]:
# %% [3] Load the authors' own code verbatim from their notebooks; stub the Logger (it writes ../../logs on every call)
def code_cells(nb):
    return ["".join(c["source"]) for c in json.load(open(nb))["cells"] if c["cell_type"] == "code"]

class Logger:
    error_log = info_log = staticmethod(lambda s: None)

G = {"Logger": Logger}
for src in code_cells(f"{ROOT}/_02_data_collection/dataset_post_filtering/remove_all_comments.ipynb"):
    if "def remove_all_comments" in src:
        exec(src, G)
for src in code_cells(f"{ROOT}/_03_feature_extraction/extract_main_features.ipynb"):
    if src.lstrip().startswith("import") or "default_values =" in src or "class CodeMetricsExtractor" in src:
        exec(src, G)
remove_all_comments, CodeMetricsExtractor = G["remove_all_comments"], G["CodeMetricsExtractor"]

In [ ]:
# %% [4] Paper protocol (rq1_main_classification.ipynb): shuffle, GroupKFold(10) by problem, default XGBClassifier,
# P/R/F1 from pooled confusion counts, "AUC-ROC" = mean over folds of roc_auc_score on HARD 0/1 predictions.
# true_auroc = AUROC on pooled out-of-fold P(AI), which is what modules/metrics.py computes.
def cv(df, seed, tree_method):
    d = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    groups = d["filename"].str.split("_").str[1].astype("category").cat.codes
    X, y = d.drop(columns=["filename", "label"]), d["label"].astype(int)
    oof, label_auc = np.zeros(len(d)), []
    for tr, te in GroupKFold(n_splits=10).split(X, y, groups):
        clf = xgboost.XGBClassifier(tree_method=tree_method).fit(X.iloc[tr], y.iloc[tr])
        oof[te] = clf.predict_proba(X.iloc[te])[:, 1]
        label_auc.append(roc_auc_score(y.iloc[te], clf.predict(X.iloc[te])))
    pred = (oof > 0.5).astype(int)                      # XGBClassifier.predict rule
    tp, fn = int(((pred == 1) & (y == 1)).sum()), int(((pred == 0) & (y == 1)).sum())
    tn, fp = int(((pred == 0) & (y == 0)).sum()), int(((pred == 1) & (y == 0)).sum())
    p, r = tp / (tp + fp), tp / (tp + fn)
    return dict(TP=tp, FN=fn, TN=tn, FP=fp, f1=2 * p * r / (p + r), paper_auc=np.mean(label_auc),
                true_auroc=roc_auc_score(y, oof), miss_rate=fn / (tp + fn), fpr=fp / (fp + tn))

In [ ]:
# %% [5] Reproduce Table 4 from the shipped feature CSVs. Targets = counts printed in the shipped notebooks.
TARGET = {"main136": dict(TP=728, FN=70, TN=726, FP=72, f1=0.91, paper_auc=0.91),     # Table 4 "All"
          "nongame134": dict(TP=707, FN=91, TN=713, FP=85, f1=0.89, paper_auc=0.89),  # Table 4 "Non-Gameable"
          "halstead140": dict(TP=735, FN=63, TN=723, FP=75, f1=0.91, paper_auc=0.91)} # §4 RQ1 text only
CSVS = {"main136": "rq1_main_features.csv", "nongame134": "rq2_non_gameable_features.csv",
        "halstead140": "rq1_with_halstead_features.csv"}
data = {k: pd.read_csv(f"{FEAT}/{v}") for k, v in CSVS.items()}
SEEDS = range(5)   # the paper's shuffle is unseeded; fold membership does not depend on it (all groups have size 4)
res = pd.DataFrame([dict(variant=k, tree_method=tm, seed=s, **cv(df, s, tm))
                    for k, df in data.items() for tm in ["exact", "hist"] for s in SEEDS])
COLS = ["TP", "FN", "TN", "FP", "f1", "paper_auc", "true_auroc", "miss_rate", "fpr"]
print(res.groupby(["variant", "tree_method"])[COLS].mean().round(3).to_string())
print(res.groupby(["variant", "tree_method"])["f1"].std().round(4).rename("f1_std").to_string())
for k, t in TARGET.items():
    sub = res[(res.variant == k) & (res.tree_method == "exact")]
    counts = (sub[["TP", "FN", "TN", "FP"]] == pd.Series({c: t[c] for c in ["TP", "FN", "TN", "FP"]})).all(axis=1).mean()
    ok = abs(sub.f1.mean() - t["f1"]) <= 0.02 and abs(sub.paper_auc.mean() - t["paper_auc"]) <= 0.02
    print(f"{k}: exact counts in {counts:.0%} of seeds | F1 {sub.f1.mean():.3f} vs {t['f1']} | "
          f"paper-AUC {sub.paper_auc.mean():.3f} vs {t['paper_auc']} | {'PASS' if ok else 'FAIL'}")

In [ ]:
# %% [6] score() path on THIS Python: strip comments + extract from raw code, diff vs the CSV, then re-run CV
MAIN_COLS = [c for c in data["main136"].columns if c not in ("filename", "label")]
NONGAME_COLS = [c for c in MAIN_COLS if c not in ("whiteSpaceRatio", "emptyLinesDensity")]

def featurize(code):
    stripped, _ = remove_all_comments(code)
    m = CodeMetricsExtractor(stripped).metrics            # {} when the stripped code does not parse
    return {c: m.get(c, 0) for c in MAIN_COLS} if m else None   # absent node types/keywords -> 0, as get_uniform_metrics

recs, failed = {}, []
for fn in data["main136"].filename:
    try:
        f = featurize(open(f"{COMBINED}/{fn}", encoding="utf-8").read())
    except Exception:
        f = None
    if f:
        recs[fn] = f
    else:
        failed.append(fn)
new = pd.DataFrame.from_dict(recs, orient="index")[MAIN_COLS]
old = data["main136"].set_index("filename").loc[new.index, MAIN_COLS]
a, b = new.to_numpy(float), old.to_numpy(float)
same = np.isclose(a, b, atol=1e-6) | (np.isnan(a) & np.isnan(b))
print(f"extraction failures: {len(failed)} | rows identical on all 136 features: {same.all(axis=1).mean():.1%}")
print("mismatching cells per column:", pd.Series((~same).sum(axis=0), index=MAIN_COLS).loc[lambda s: s > 0].to_dict())
new_df = new.assign(filename=new.index, label=new.index.str.startswith("ai").astype(int)).reset_index(drop=True)
for k, cols in [("main136", MAIN_COLS), ("nongame134", NONGAME_COLS)]:
    r = pd.DataFrame([cv(new_df[cols + ["filename", "label"]], s, "exact") for s in SEEDS])
    print(f"recomputed {k}: " + ", ".join(f"{c}={r[c].mean():.3f}" for c in COLS))

In [ ]:
# %% [7] Final detector (train on all rows, recomputed features) + score() smoke test
final = xgboost.XGBClassifier(tree_method="exact").fit(new[MAIN_COLS], new.index.str.startswith("ai").astype(int))

def score(codes):
    rows = []
    for code in codes:
        try:
            rows.append(featurize(code))
        except Exception:
            rows.append(None)
    p = final.predict_proba(pd.DataFrame([r or {} for r in rows], columns=MAIN_COLS).astype(float))[:, 1]
    return [float(pi) if r else 0.5 for pi, r in zip(p, rows)]   # unparseable -> 0.5 (counted as a miss at 0.5)

print(score(["n=int(input())\nprint(sum(map(int,input().split())))",
             "def main():\n    n = int(input())\n    print(n * 2)\n\n\nif __name__ == '__main__':\n    main()\n",
             "x = foo(a,  # comment\n        b)\n"]))   # last one: the authors' stripper breaks it -> 0.5